In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLNetTokenizer, XLNetModel
print("Libraries and Utils Imported")

Libraries and Utils Imported


## Reading the Data

In [2]:
df = pd.read_csv(r"cleaned_political_bias_data (1).csv")
df.sample(3)

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
1906,635,us weekly jobless claims rise for a second str...,the number of americans who filed for unemploy...,center,center,a a million more americans file for unemployment,economy and jobs,the number of americans who filed for unemploy...
2238,746,veteran bar owner commits suicide week after s...,an omaha bar owner who was a two tour veteran ...,right,conservative,bar owner charged with murder during nebraska ...,violence in america,an omaha bar owner who was a two tour veteran ...
7559,2522,republicans vote to acquit trump on impeachmen...,senate republicans voted against convicting do...,lean left,liberal,senate acquits trump in second impeachment trial,impeachment,senate republicans voted against convicting do...


In [3]:
print("Total Samples:",len(df))
print(df['Stance'].value_counts())
print(df['Label'].value_counts())

Total Samples: 9190
Stance
center        3054
lean left     2237
lean right    2088
right          974
left           829
mixed            6
not rated        2
Name: count, dtype: int64
Label
liberal         3066
conservative    3062
center          3062
Name: count, dtype: int64


Hence Data is Evenly Distributed, No Imbalance of Data

## Setting up Stance to Ideology

This is setup since an Encoder is trained to extract contextual embeddings best from sentences.

In [4]:
## Setting up Stance to Ideology as a dictionary
stance_to_ideology = {
    "left" : "far left leaning ideology",
    'lean left' : 'slightly left leaning ideology',
    'center'    : 'centrist ideology with balanced reporting',
    'lean right': 'slightly right leaning ideology',
    'right'     : 'far right leaning ideology',
    'mixed'     : 'mixed or undefined ideological leaning',
    'not rated' : 'ideology not rated or unknown',
}

## Setting Up the Encoder Configuration

In [5]:
from transformers import XLNetConfig, XLNetModel
from torch.utils.data import Dataset, DataLoader

# Initializing XLNet Tokenizer and Model
Encoder = XLNetModel.from_pretrained('xlnet-base-cased')
Tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

# Max Sequence Length (if needed more can be adjusted)
max_len = 64

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        stance_text = str(row['Stance_Ideology']) # Stringifying the Stance row
        issue_text = str(row['issue']) # Stringifying the Issue row
        topic_text = str(row['topic']) # Stringifying the Topic row

        encoding_stance = self.tokenizer(
            stance_text,
            add_special_tokens=True, # Start,Stop,Seperator tokens
            max_length=self.max_len,
            return_token_type_ids=False, # Since Single Sequences processed not needed
            padding='max_length', # All Sequences padded to max_len
            return_attention_mask=True, # mask the padded sequences
            return_tensors='pt', # Return Pytorch Tensors
            truncation=True # We Truncate if len>max_len of the sentence/sequence
        )

        encoding_issue = self.tokenizer(
            issue_text,
            add_special_tokens=True, # Start,Stop,Seperator tokens
            max_length=self.max_len,
            return_token_type_ids=False, # Since Single Sequences processed not needed
            padding='max_length', # All Sequences padded to max_len
            return_attention_mask=True, # mask the padded sequences
            return_tensors='pt', # Return Pytorch Tensors
            truncation=True # We Truncate if len>max_len of the sentence/sequence
        )

        encoding_topic = self.tokenizer(
            topic_text,
            add_special_tokens=True, # Start,Stop,Seperator tokens
            max_length=self.max_len,
            return_token_type_ids=False, # Since Single Sequences processed not needed
            padding='max_length', # All Sequences padded to max_len
            return_attention_mask=True, # mask the padded sequences
            return_tensors='pt', # Return Pytorch Tensors
            truncation=True # We Truncate if len>max_len of the sentence/sequence
        )

        return {
            'input_ids_stance': encoding_stance['input_ids'].flatten(),
            'attention_mask_stance': encoding_stance['attention_mask'].flatten(),
            'input_ids_issue': encoding_issue['input_ids'].flatten(),
            'attention_mask_issue': encoding_issue['attention_mask'].flatten(),
            'input_ids_topic': encoding_topic['input_ids'].flatten(),
            'attention_mask_topic': encoding_topic['attention_mask'].flatten()
        }

# Returning a Dictionary of Attention Mask Matrices & Embedded Matrices from XLNet Tokenizer

print("CustomDataset class defined.")

CustomDataset class defined.


### Encoder with Linear Projection Block
Here is the implementation of Encoder(XLNet) model which processes the three text inputs separately and then projects the concatenated embeddings to 256 dimensions using `LinearProjectionBlock`.

### Linear Projection Block

In [7]:
class LinearProjectionBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dim:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(p=dropout_rate))
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        layers.append(nn.BatchNorm1d(output_dim))
        # layers.append(nn.ReLU(inplace = True)) # Last-Layer does not use RELU as we need feature vector
        self.projection = nn.Sequential(*layers)

    def forward(self, x):
        return self.projection(x)

print("Linear Projection Block defined.")

Linear Projection Block defined.


### The Complete Triple Stance/Topic/Issue Encoder

In [8]:
class Encoder(nn.Module):

  def __init__(self,model_name,hidden_dim,output_dim,dropout_rate=0.5):
    super().__init__()
    self.xlnet = XLNetModel.from_pretrained(model_name)
    hidden_size = self.xlnet.config.hidden_size #768 for xlnet-base-cased
    self.projection = LinearProjectionBlock(
        input_dim=hidden_size,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        dropout_rate=dropout_rate,
    )

  def forward(self,input_ids,attention_mask):
    outputs = self.xlnet(input_ids=input_ids,attention_mask=attention_mask)
    hidden = outputs.last_hidden_state
    pooled = hidden[:, -1, :]
    return self.projection(pooled)

# Constructor takes: model_name, hidden_dim (intermediate sizes inside LinearProjectionBlock,
# e.g. [512]), output_dim (256), and dropout_rate.
# input_dim (768) is read automatically from XLNet's config — not passed in.
# Then for Forward Propagation, We use input ids and attention masks which we get from Custom Dataset and
# We Return a (Batchsize x 256) dimension Vector

print("Encoder defined.")

Encoder defined.


## Now the Combined Stance-Topic-Issue XLNet Encoders

In [9]:
class TripleEncoder(nn.Module):

  def __init__(self,model_name,hidden_dim,output_dim=256,dropout_rate=0.5):
    super().__init__()

    encoder_kwargs = {
        "model_name":model_name,
        "hidden_dim":hidden_dim,
        "output_dim":output_dim,
        "dropout_rate":dropout_rate
    }

    print("Stance XLNet Encoder")
    self.stance_encoder = Encoder(**encoder_kwargs)
    print("Issue XLNet Encoder")
    self.issue_encoder = Encoder(**encoder_kwargs)
    print("Topic XLNet Encoder")
    self.topic_encoder = Encoder(**encoder_kwargs)

    print("All Encoders Initialized")
    print("Output dimensions=",output_dim)
    print("Total Parameters:",sum(p.numel() for p in self.parameters()))


  def forward(self,batch):
    stance_vec = self.stance_encoder(
        input_ids=batch['input_ids_stance'],
        attention_mask=batch['attention_mask_stance']
    )
    issue_vec = self.issue_encoder(
        input_ids=batch['input_ids_issue'],
        attention_mask=batch['attention_mask_issue']
    )
    topic_vec = self.topic_encoder(
        input_ids=batch['input_ids_topic'],
        attention_mask=batch['attention_mask_topic']
    )
    return {
        'stance_vec':stance_vec,  # (B, 256)
        'issue_vec':issue_vec,  # (B, 256)
        'topic_vec':topic_vec,  # (B, 256)
    }


print("TripleEncoder Class Instantiated!")
# TripleEncoder wraps three independent XLNetEncoders — one each for Stance, Issue and Topic.
# Each encoder runs its own XLNet backbone followed by the LinearProjectionBlock.
# Forward pass takes a batch from CustomDataset and passes each attribute
# through its corresponding encoder.
# Returns a dictionary of three independent 256-dimensional context vectors:
# stance_vec (B, 256), issue_vec (B, 256), topic_vec (B, 256)

TripleEncoder Class Instantiated!


### Dataset & DataLoader

In [10]:
# Create Dataset and DataLoader
dataset    = CustomDataset(df, Tokenizer, max_len)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Instantiate the model
model = TripleEncoder(
    model_name   = 'xlnet-base-cased',  # string — each XLNetEncoder loads its own backbone
    hidden_dim   = [512],               # LinearProjectionBlock: 768 → 512 → 256
    output_dim   = 256,
    dropout_rate = 0.5,
)

print("Model instantiated:")
print(model)

Stance XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Issue XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Topic XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All Encoders Initialized
Output dimensions= 256
Total Parameters: 351734784
Model instantiated:
TripleEncoder(
  (stance_encoder): Encoder(
    (xlnet): XLNetModel(
      (word_embedding): Embedding(32000, 768)
      (layer): ModuleList(
        (0-11): 12 x XLNetLayer(
          (rel_attn): XLNetRelativeAttention(
            (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (ff): XLNetFeedForward(
            (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (layer_1): Linear(in_features=768, out_features=3072, bias=True)
            (layer_2): Linear(in_features=3072, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (activation_function): GELUActivation()
          )
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (projection): LinearProject

## My Device Details

In [11]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(DEVICE)
print(f'Running on: {DEVICE}')

Running on: cpu


# Inference — Extract 256-dim Context Vectors for Full Dataset

In [ ]:
'''
model.eval()  # Disable dropout & set BatchNorm to eval mode

# Storage for all three streams across all batches
all_stance_vecs = []
all_issue_vecs  = []
all_topic_vecs  = []

with torch.no_grad():  # No gradients needed during inference — saves memory
    for i, batch in enumerate(dataloader):

        # Move batch tensors to device
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        # Forward pass through XLNetTripleEncoder
        outputs = model(batch)

        # Move results back to CPU immediately to free GPU memory
        all_stance_vecs.append(outputs['stance_vec'].cpu())
        all_issue_vecs.append(outputs['issue_vec'].cpu())
        all_topic_vecs.append(outputs['topic_vec'].cpu())

        if (i + 1) % 10 == 0:
            print(f'  Processed {(i + 1) * dataloader.batch_size} / {len(dataset)} samples...')

# Concatenate all batches → final matrices for the full dataset
stance_matrix = torch.cat(all_stance_vecs, dim=0)  # (N, 256)
issue_matrix  = torch.cat(all_issue_vecs,  dim=0)  # (N, 256)
topic_matrix  = torch.cat(all_topic_vecs,  dim=0)  # (N, 256)

print('\nInference Complete!')
print(f'Stance Matrix : {stance_matrix.shape}')  # (total_samples, 256)
print(f'Issue Matrix  : {issue_matrix.shape}')   # (total_samples, 256)
print(f'Topic Matrix  : {topic_matrix.shape}')   # (total_samples, 256)
'''

### Verify — Sample Vector Statistics

In [ ]:
# Quick sanity check on the first sample of each stream
'''
print(f"{'Stream':<14} {'Shape':<18} {'Norm':>8} {'Mean':>8} {'Std':>8}")
print('-' * 60)
for name, matrix in [('stance_matrix', stance_matrix), ('issue_matrix', issue_matrix), ('topic_matrix', topic_matrix)]:
    v0 = matrix[0]
    print(f"{name:<14} {str(tuple(matrix.shape)):<18} "
          f"{v0.norm().item():8.4f} "
          f"{v0.mean().item():8.4f} "
          f"{v0.std().item():8.4f}")
'''

### Functions to get Sentence wide Embeddings of Issue/Topic/Stance


As Per Design, here is an implementation of 3 Functions where we Get Encoded Embeddings from a single Sentence for Topic/Issue and Stance.

In [18]:
import torch

model = TripleEncoder(
    model_name   = 'xlnet-base-cased',  # string — each XLNetEncoder loads its own backbone
    hidden_dim   = [512],               # LinearProjectionBlock: 768 → 512 → 256
    output_dim   = 256,
    dropout_rate = 0.5,
)

# Model in Eval Mode
model.eval()

## Encoding the Text where it takes into input the text and returns Input Ids and Attention Mask Vector
def _encode_text(text):
    encoding = Tokenizer(
        str(text),
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
        truncation=True
    )

    input_ids = encoding['input_ids']          # (1, max_len)
    attention_mask = encoding['attention_mask']# (1, max_len)

    return input_ids, attention_mask

Stance XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Issue XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Topic XLNet Encoder


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.bias   | UNEXPECTED |  | 
lm_loss.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All Encoders Initialized
Output dimensions= 256
Total Parameters: 351734784


In [19]:

# Returns a 1 x 256 Dimension Vector after taking into input a sentence
def GetIssueVector(text):
    input_ids, attention_mask = _encode_text(text)

    with torch.no_grad():
        vec = model.issue_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )  # (1, 256)

    return vec

def GetTopicVector(text):
    input_ids, attention_mask = _encode_text(text)

    with torch.no_grad():
        vec = model.topic_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )  # (1, 256)

    return vec


def GetStanceVector(text):
    input_ids, attention_mask = _encode_text(text)

    with torch.no_grad():
        vec = model.stance_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )  # (1, 256)

    return vec

In [20]:
issue_vec  = GetIssueVector("Climate change policies")
topic_vec  = GetTopicVector("Global warming")
stance_vec = GetStanceVector("We must reduce carbon emissions")

print(issue_vec)   # torch.Size([1, 256])
# print(topic_vec)   # torch.Size([1, 256])
# print(stance_vec)   # torch.Size([1, 256])

tensor([[-1.2113, -1.0163,  0.1302,  0.5603,  1.1441, -1.3862,  0.2469, -0.6238,
         -0.4848, -1.8039,  0.9892,  0.9187, -0.9075, -0.8470, -0.0042,  0.1245,
          0.3849, -0.2909, -1.0955, -0.1676,  1.3726,  0.2034,  0.4549, -1.3820,
          1.1199,  1.6671, -0.1222, -1.6583, -0.6679,  0.9793,  1.1697,  0.2880,
          1.0387,  0.3197,  1.0597, -0.3696,  0.1962,  1.8041,  0.5303, -0.8840,
          1.7304, -0.6750, -2.1983,  1.3056, -0.2857,  0.3541,  1.0825, -0.6599,
          0.4014,  0.1895,  0.6557,  2.5531, -0.9807,  1.3001,  1.1282,  2.0212,
         -0.1461,  0.5983, -2.6555, -1.1301, -1.1841,  1.1919, -0.0773, -0.3844,
         -0.0187, -1.6887,  0.8517,  1.4782,  0.1536,  1.6223,  0.4263, -0.5594,
          0.8922, -0.8657,  0.5625, -2.0988, -0.6573,  0.7146, -0.4324, -0.4159,
         -0.9193,  0.6621, -1.2312,  1.4522, -1.3274,  1.0836,  0.3599, -0.8958,
          2.0130,  0.3636, -1.0548, -0.1927,  0.4741,  0.2045,  0.8561,  0.8697,
          2.4031, -0.2742,  